In [1]:
import pandas as pd
import numpy as np

# Define relative data paths
orders_path = '../data/raw/olist_orders_dataset.csv'
items_path = '../data/raw/olist_order_items_dataset.csv'
customers_path = '../data/raw/olist_customers_dataset.csv'
payments_path = '../data/raw/olist_order_payments_dataset.csv'

# Load raw datasets
df_orders = pd.read_csv(orders_path)
df_items = pd.read_csv(items_path)
df_customers = pd.read_csv(customers_path)
df_payments = pd.read_csv(payments_path)

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [2]:
# Convert order timestamp fields to datetime objects
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    df_orders[col] = pd.to_datetime(df_orders[col], errors='coerce')

# Convert shipping limit date in order items
df_items['shipping_limit_date'] = pd.to_datetime(df_items['shipping_limit_date'], errors='coerce')

# Verify parsed data types
df_orders[date_columns].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [3]:
# Check order status distribution
print("Initial record count:", len(df_orders))

# Retain completed/delivered orders for baseline sales metrics
df_orders_clean = df_orders[df_orders['order_status'] == 'delivered'].copy()

# Remove records missing customer delivery timestamp
df_orders_clean = df_orders_clean.dropna(subset=['order_delivered_customer_date'])

print("Cleaned delivered orders count:", len(df_orders_clean))

Initial record count: 99441
Cleaned delivered orders count: 96470


In [4]:
# Identify potential data entry anomalies (delivery before purchase)
invalid_delivery_mask = df_orders_clean['order_delivered_customer_date'] < df_orders_clean['order_purchase_timestamp']
anomalous_records = df_orders_clean[invalid_delivery_mask]

print(f"Anomalous delivery timestamp records: {len(anomalous_records)}")

# Exclude anomalous records if any exist
df_orders_clean = df_orders_clean[~invalid_delivery_mask]
print("Final valid delivered orders count:", len(df_orders_clean))

Anomalous delivery timestamp records: 0
Final valid delivered orders count: 96470
